In [0]:
# %pip install sentence-transformers scikit-learn numpy
# dbutils.library.restartPython()

In [0]:
# from sentence_transformers import SentenceTransformer
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

In [0]:
# print("Loading embedding model...")

# embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")

# print("Embedding model loaded.")

In [0]:
# INTENT_PROTOTYPES = {

#     "legal_qa": [
#         "What law applies if I was cheated online?",
#         "How do I file a complaint for fraud?",
#         "What punishment exists for theft?",
#         "What section of law applies to cyber crime?"
#     ],

#     "scheme_query": [
#         "Which government schemes can help poor families?",
#         "Am I eligible for any government welfare schemes?",
#         "Is there any financial help scheme from government?",
#         "Government scheme for education support"
#     ],

#     "ipc_bns_comparison": [
#         "What is the difference between IPC and BNS?",
#         "Compare IPC sections with BNS sections",
#         "What changed from IPC to Bharatiya Nyaya Sanhita",
#         "Explain IPC vs BNS law changes"
#     ],

#     "legal_summarization": [
#         "Summarize this legal section",
#         "Explain this law in simple words",
#         "Give a short explanation of this legal clause",
#         "Simplify this legal document"
#     ]

# }

In [0]:
# print("Precompute prototype embeddings")

# protype_embeddings = {}
# for intent, ex in INTENT_PROTOTYPES.items():
#     formatted_ex = [f"query: {e}" for e in ex]
#     embeddings = embedding_model.encode(formatted_ex, normalize_embeddings=True)
#     protype_embeddings[intent] = embeddings

# print("Prototype embeddings computed")

In [0]:
# %skip
# def sementic_intent_classification(query, threshold = 0.45):
#     formatted_query = f"query: {query}"
#     intent_scores = {}
#     query_embeddings = embedding_model.encode(formatted_query, normalize_embeddings=True).reshape(1, -1)
#     for intent, prototypes in protype_embeddings.items():
#         cosine_similarities = cosine_similarity(query_embeddings, prototypes)[0]
#         intent_scores[intent] = float(np.max(cosine_similarities))

#     sorted_scores = sorted(intent_scores.items(), key=lambda x: x[1], reverse=True)
#     detected_intents = [
#         intent for intent, score in sorted_scores if score >= threshold
#     ]

#     if len(detected_intents) == 0:
#         detected_intents = ["legal_qa"]

#     return {
#         "query": query,
#         "intents": detected_intents,
#         "scores": intent_scores
#     }

In [0]:
# test_queries = [
#     "If someone forwards a message in a group that later turns out to cause panic or harm, but they didn’t create it themselves, can they still get into serious trouble, and does it matter whether they knew it was false or not?",

#     "can my collage suspend me for conducting an event for taking bribe",

#     "Explain BNS section for theft",

#     "What changed from IPC to BNS",

#     "Summarize this legal section for me"

# ]

# for q in test_queries:
    
#     result = sementic_intent_classification(q)

#     print("\nQuery:", q)
#     print("Detected Intents:", result["intents"])
#     print("Scores:", result["scores"])

RAG PIPLINE

In [0]:
%pip install langgraph langchain langchain-community

In [0]:
# from typing import TypedDict,List, Dict, Any
# from langgraph.graph import StateGraph, END

In [0]:
# class AgentState(TypedDict, total=False):
#     query: str
#     intents: List[str]
#     intent_scores: Dict[str, float]
#     conversation_history: List[str]
#     case_summary: str
#     answer_parts: Dict[str, str]
#     sources: List[Dict[str, Any]]
#     action_pack: str
#     final_answer: str

In [0]:
# def query_understanding_node(state: AgentState):
#     query = state["query"]
#     conversation_history = state.get("conversation_history", [])
#     case_summary = state.get("case_summary", "")

#     memory_hint = ""
#     if case_summary:
#         memory_hint += f"Case Summary: {case_summary}\n"
#     if conversation_history:
#         memory_hint += "Conversation History:\n" + "\n".join(conversation_history[-6:]) + "\n"

#     enriched_query = f"{memory_hint}\nUser Query: {query}".strip()

#     intent_result = semantic_intent_classification(enriched_query)

#     return {
#         "intents": intent_result.get("intents", ["legal_qa"]),
#         "intent_scores": intent_result.get("scores", {})
#     }

In [0]:
# def route_query(state: AgentState):
#     intents = state.get("intents", [])
#     if not intents:
#         return "legal_only"

#     if len(intents) > 1:
#         return "multi_intent"

#     intent = intents[0]

#     if intent == "legal_qa":
#         return "legal_only"
#     if intent == "scheme_query":
#         return "scheme_only"
#     if intent == "ipc_bns_comparison":
#         return "comparison_only"
#     if intent == "legal_summarization":
#         return "summary_only"

#     return "legal_only"

In [0]:
# def legal_rag_node(state: AgentState):

#     query = state["query"]
#     case_summary = state.get("case_summary", "")

#     rag_query = query

#     if case_summary:
#         rag_query = case_summary + "\n" + query

#     answer = legal_rag_pipeline(rag_query)

#     return {
#         "responses": {"legal": answer}
#     }

# def scheme_rag_node(state: AgentState):

#     query = state["query"]
#     case_summary = state.get("case_summary", "")

#     rag_query = query

#     if case_summary:
#         rag_query = case_summary + "\n" + query

#     answer = scheme_rag_pipeline(rag_query)

#     return {
#         "responses": {"legal": answer}
#     } 
# def comparison_node(state: AgentState):

#     query = state["query"]

#     answer = f"IPC vs BNS comparison result for: {query}"

#     return {
#         "comparison_answer": answer
#     }

# def summarization_node(state: AgentState):

#     query = state["query"]

#     answer = f"Summary of legal text: {query}"

#     return {
#         "summary_answer": answer
#     }

In [0]:
# def combine_answers_node(state: AgentState):
#     parts = state.get("answer_parts", {})
#     blocks = []

#     if parts.get("legal_qa"):
#         blocks.append("## Legal Answer\n" + parts["legal_qa"])

#     if parts.get("scheme_query"):
#         blocks.append("## Scheme Match\n" + parts["scheme_query"])

#     if parts.get("ipc_bns_comparison"):
#         blocks.append("## IPC → BNS Comparison\n" + parts["ipc_bns_comparison"])

#     if parts.get("legal_summarization"):
#         blocks.append("## Summary\n" + parts["legal_summarization"])

#     action_pack = state.get("action_pack", "")
#     if action_pack:
#         blocks.append(action_pack)

#     sources = dedupe_sources(state.get("sources", []))
#     if sources:
#         src_lines = []
#         for i, s in enumerate(sources, start=1):
#             src_lines.append(
#                 f"{i}. {s.get('title', 'Unknown')} | page {s.get('page', 'NA')} | {s.get('source_file', 'NA')}"
#             )
#         blocks.append("### Sources\n" + "\n".join(src_lines))

#     return {
#         "final_answer": "\n\n".join(blocks)
#     }

In [0]:
# def memory_update_node(state: AgentState):
#     conversation_history = list(state.get("conversation_history", []))
#     case_summary = state.get("case_summary", "")

#     query = state.get("query", "")
#     final_answer = state.get("final_answer", "")

#     conversation_history.append(f"User: {query}")
#     conversation_history.append(f"Assistant: {final_answer}")
#     conversation_history = conversation_history[-10:]

#     # lightweight summary for now
#     updated_summary = f"{case_summary}\nQ: {query}\nA: {final_answer}".strip()
#     updated_summary = updated_summary[-2000:]

#     return {
#         "conversation_history": conversation_history,
#         "case_summary": updated_summary
#     }

In [0]:
# workflow = StateGraph(AgentState)

# workflow.add_node("query_understanding", query_understanding_node)
# workflow.add_node("legal_only", legal_rag_node)
# workflow.add_node("scheme_only", scheme_node)
# workflow.add_node("comparison_only", comparison_only_node)
# workflow.add_node("summary_only", summary_only_node)
# workflow.add_node("multi_intent", multi_intent_node)
# workflow.add_node("combine_answers", combine_answers_node)
# workflow.add_node("memory_update", memory_update_node)

# workflow.set_entry_point("query_understanding")

# workflow.add_conditional_edges(
#     "query_understanding",
#     route_query,
#     {
#         "legal_only": "legal_only",
#         "scheme_only": "scheme_only",
#         "comparison_only": "comparison_only",
#         "summary_only": "summary_only",
#         "multi_intent": "multi_intent",
#     }
# )

# workflow.add_edge("legal_only", "combine_answers")
# workflow.add_edge("scheme_only", "combine_answers")
# workflow.add_edge("comparison_only", "combine_answers")
# workflow.add_edge("summary_only", "combine_answers")
# workflow.add_edge("multi_intent", "combine_answers")

# workflow.add_edge("combine_answers", "memory_update")
# workflow.add_edge("memory_update", END)

# graph = workflow.compile()

In [0]:
# result = app.invoke({

#     "query": "I was cheated in an online payment what law applies and is there any government scheme for help"

# })

# print(result["final_answer"])

In [0]:
"""
query_understanding.py
======================

This module implements a simple intent classification function for the
Nyaya‑Sahayak assistant.  It uses a multilingual sentence transformer to
embed queries and compares them against a handful of prototype examples
for each intent via cosine similarity.  The intent names correspond to
the different retrieval pipelines (legal Q&A, scheme queries, IPC/BNS
comparisons, and summarisation).

The classifier is lightweight and designed to run on CPU.  It loads the
model once at module import time and pre‑computes prototype embeddings
for faster inference.  To add new intents, extend the INTENT_PROTOTYPES
dictionary with additional examples.
"""

from typing import Dict, List, Tuple
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


# -----------------------------------------------------------------------------
# Prototype examples for each intent.  Feel free to expand these lists to
# improve classification accuracy.  The order of examples does not matter.
# -----------------------------------------------------------------------------
INTENT_PROTOTYPES: Dict[str, List[str]] = {
    "legal_qa": [
        "What law applies if I was cheated online?",
        "How do I file a complaint for fraud?",
        "What punishment exists for theft?",
        "What section of law applies to cyber crime?",
    ],
    "scheme_query": [
        "Which government schemes can help poor families?",
        "Am I eligible for any government welfare schemes?",
        "Is there any financial help scheme from government?",
        "Government scheme for education support",
    ],
    "ipc_bns_comparison": [
        "What is the difference between IPC and BNS?",
        "Compare IPC sections with BNS sections",
        "What changed from IPC to Bharatiya Nyaya Sanhita",
        "Explain IPC vs BNS law changes",
    ],
    "legal_summarization": [
        "Summarize this legal section",
        "Explain this law in simple words",
        "Give a short explanation of this legal clause",
        "Simplify this legal document",
    ],
}


# -----------------------------------------------------------------------------
# Load the embedding model once.  The intfloat/multilingual‑e5‑small model
# supports 100+ languages and is small enough to run efficiently on CPU.  By
# normalising the embeddings we can use cosine similarity directly.  On
# import this may take a few seconds; subsequent calls reuse the same model.
# -----------------------------------------------------------------------------
print("Loading intent classification embedding model…")
_embedding_model: SentenceTransformer = SentenceTransformer(
    "intfloat/multilingual-e5-small"
)
print("Intent classification model loaded.")


# -----------------------------------------------------------------------------
# Precompute embeddings for each prototype example.  We prefix each example
# with "query: " to match the format recommended by the model authors for
# retrieval tasks.  The resulting dictionary maps each intent to a matrix of
# shape (num_examples, embedding_dim).  Embeddings are normalised so that
# cosine similarity reduces to a dot product.
# -----------------------------------------------------------------------------
_prototype_embeddings: Dict[str, np.ndarray] = {}
for intent, examples in INTENT_PROTOTYPES.items():
    formatted_examples = [f"query: {ex}" for ex in examples]
    embeddings = _embedding_model.encode(
        formatted_examples, normalize_embeddings=True
    )
    _prototype_embeddings[intent] = embeddings


def semantic_intent_classification(query: str, threshold: float = 0.45) -> Dict[str, object]:
    """Classify a user query into one or more intents.

    The query is embedded using the sentence transformer and compared
    against the prototype examples for each intent.  An intent is
    considered a match if the maximum cosine similarity across its
    prototypes exceeds the given threshold.  If no intent exceeds
    the threshold, the function returns ["legal_qa"] as a default.

    Args:
        query: The user query string.
        threshold: Cosine similarity threshold for detecting an intent.

    Returns:
        A dictionary containing the original query, the list of detected
        intents, and a mapping of all intent scores.
    """
    formatted_query = f"query: {query}"
    # Embed the query
    query_vec = _embedding_model.encode(
        formatted_query, normalize_embeddings=True
    ).reshape(1, -1)

    intent_scores: Dict[str, float] = {}
    detected_intents: List[str] = []
    # Compute the maximum cosine similarity for each intent
    for intent, proto_vecs in _prototype_embeddings.items():
        sims = cosine_similarity(query_vec, proto_vecs)[0]
        max_sim = float(np.max(sims))
        intent_scores[intent] = max_sim
        if max_sim >= threshold:
            detected_intents.append(intent)

    # Default to legal_qa if nothing matches
    if not detected_intents:
        detected_intents = ["legal_qa"]

    return {
        "query": query,
        "intents": detected_intents,
        "scores": intent_scores,
    }


def get_intents(query: str, threshold: float = 0.45) -> List[str]:
    """Return just the list of intents for a given query."""
    return semantic_intent_classification(query, threshold)["intents"]


__all__ = [
    "semantic_intent_classification",
    "get_intents",
    "INTENT_PROTOTYPES",
]